# 🍎 Pixels-to-Macros — Food Segmentation Training

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8  
Then open Cell 9 in a **second browser tab** for live metrics.

**Session reset / resume:** Re-run cells 2 → 3 → 7 → 8 — training picks up from the last checkpoint automatically.

> Outputs are saved to **Google Drive** (`pixels-to-macros/`) so nothing is lost when Colab disconnects.


## Cell 1 — Verify GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Cell 2 — Mount Google Drive (saves checkpoints across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib

# ── Change this if your Drive folder has a different name ─────────
OUTPUT_DIR = pathlib.Path('/content/drive/MyDrive/pixels-to-macros')
# ─────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ckpt = OUTPUT_DIR / 'last_checkpoint.pth'
best = OUTPUT_DIR / 'best.pth'

print(f'Output dir : {OUTPUT_DIR}')
if ckpt.exists():
    print(f'Checkpoint : {ckpt.stat().st_size / 1e6:.1f} MB  ← will auto-resume from here')
else:
    print('Checkpoint : none — will start fresh')
if best.exists():
    print(f'Best model : {best.stat().st_size / 1e6:.1f} MB')


## Cell 3 — Clone repo

In [ ]:

import pathlib, subprocess

%cd /content

repo = pathlib.Path('/content/pixels-to-macros')

# Remove broken directory (exists but no .git)
if repo.exists() and not (repo / '.git').exists():
    print('Removing broken directory...')
    subprocess.run(['rm', '-rf', str(repo)], check=True)

if not repo.exists():
    !git clone https://github.com/JormayBusso/pixels-to-macros.git /content/pixels-to-macros

%cd /content/pixels-to-macros

# Force-sync to latest remote (fixes stale .py / __pycache__ issues)
!git fetch --all
!git reset --hard origin/dev

# Clear any cached .pyc that could shadow updated source files
!rm -rf training/__pycache__

print('\n=== training/ ===')
!ls training/


## Cell 4 — Install requirements

> This installs a CUDA-enabled PyTorch first, then the rest of `training/requirements.txt`.
> `coremltools` is excluded (CoreML export is macOS-only; do that step on your Mac later).

In [ ]:

# Install segmentation-models-pytorch and other training deps.
# We keep Colab's pre-installed torch (2.10+) and skip coremltools (macOS-only).
!grep -vE '^torch|^torchvision|coremltools' /content/pixels-to-macros/training/requirements.txt > /tmp/colab_reqs.txt
!pip install -q segmentation-models-pytorch timm
!pip install -q -r /tmp/colab_reqs.txt

import torch, numpy as np
print(f'torch={torch.__version__}  numpy={np.__version__}  CUDA={torch.cuda.is_available()}')


## Cell 5 — (Optional) Install SegFormer support

Only needed if you plan to train with `--model segformer`. Skip otherwise.

In [ ]:
# Uncomment to install:
# !pip install -q transformers
print('Skipped (remove comment above to install SegFormer support)')

## Cell 6 — Download FoodSeg103 dataset (~1.3 GB)

Skipped automatically if the dataset already exists.

In [ ]:

%cd /content/pixels-to-macros

import pathlib
train_dir = pathlib.Path('data/FoodSeg103/Images/img_dir/train')

if train_dir.exists() and len(list(train_dir.glob('*.jpg'))) > 100:
    print(f'Dataset already present ({len(list(train_dir.glob("*.jpg")))} train images). Skipping download.')
else:
    print('Downloading FoodSeg103 via HuggingFace (no quota limits)...')
    !pip install -q datasets
    !python scripts/download_hf_foodseg103.py
    count = len(list(train_dir.glob('*.jpg'))) if train_dir.exists() else 0
    print(f'\nDone! {count} train images ready.')


## Cell 7 — Training config

Adjust these variables before starting training.

In [ ]:
# ── Training config ── edit these before running Cell 8 ──────────
MODEL       = 'mobilenet'  # 'mobilenet' | 'resnet101' | 'segformer'
EPOCHS      = 80
BATCH_SIZE  = 8            # reduce to 4 if you get OOM
IMG_SIZE    = 512          # 512 is optimal for T4 memory + speed
LR          = 2e-4
NUM_WORKERS = 2
VAL_EVERY   = 3            # run validation every N epochs
MULTIPLIER  = 10           # augmented copies per real image per epoch
SAVE_SECS   = 180          # checkpoint to Drive at least every 3 min
SAVE_EVERY  = 100          # also checkpoint every N batches
DATA_DIR    = './data/FoodSeg103'
# OUTPUT_DIR is set in Cell 2 — do not change it here
# ─────────────────────────────────────────────────────────────────

print(f'Model     : {MODEL}')
print(f'Epochs    : {EPOCHS}  |  Batch: {BATCH_SIZE}  |  IMG: {IMG_SIZE}x{IMG_SIZE}')
print(f'Validate  : every {VAL_EVERY} epochs')
print(f'Checkpoint: every {SAVE_SECS}s or {SAVE_EVERY} batches → {OUTPUT_DIR}')


## Cell 8 — Run training 🚀

Training auto-resumes from the last checkpoint if one exists in `OUTPUT_DIR`.  
Progress bars appear in real time below this cell.

In [ ]:
%cd /content/pixels-to-macros

import pathlib
out = pathlib.Path(str(OUTPUT_DIR))
out.mkdir(parents=True, exist_ok=True)

ckpt = out / 'last_checkpoint.pth'
if ckpt.exists():
    print(f'✓ Resuming — checkpoint: {ckpt.stat().st_size / 1e6:.1f} MB')
else:
    print('Starting fresh (no checkpoint found)')

print(f'\nModel: {MODEL} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | IMG: {IMG_SIZE} | LR: {LR}\n')

!python training/train.py \
  --data_dir       {DATA_DIR} \
  --output_dir     {OUTPUT_DIR} \
  --model          {MODEL} \
  --epochs         {EPOCHS} \
  --batch_size     {BATCH_SIZE} \
  --img_size       {IMG_SIZE} \
  --lr             {LR} \
  --num_workers    {NUM_WORKERS} \
  --val_every      {VAL_EVERY} \
  --virtual_train_multiplier {MULTIPLIER} \
  --save_every_secs   {SAVE_SECS} \
  --save_every_batches {SAVE_EVERY} \
  --amp \
  --compile


## Cell 9 — Live metrics watcher

Run this in a **separate cell** while Cell 8 is running to watch epoch-level results update every 10 seconds.  
Stop it with the ■ button or `Kernel → Interrupt`.

In [ ]:
# ── Live metrics watcher ─────────────────────────────────────────
# Run this cell while training (Cell 8) is running.
# In Colab: open this notebook in a second browser tab, connect to the
# same runtime, and run this cell there for a live side-by-side view.
# ─────────────────────────────────────────────────────────────────

import time, json, pathlib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import ipywidgets as widgets

metrics_path = pathlib.Path(str(OUTPUT_DIR)) / 'metrics.json'
ckpt_path    = pathlib.Path(str(OUTPUT_DIR)) / 'last_checkpoint.pth'
best_path    = pathlib.Path(str(OUTPUT_DIR)) / 'best.pth'

def fmt(val, width, decimals=4):
    try:    return f'{float(val):>{width}.{decimals}f}'
    except: return f'{"—":>{width}}'

while True:
    clear_output(wait=True)

    if metrics_path.exists():
        try:
            data = json.loads(metrics_path.read_text())

            # ── Table ─────────────────────────────────────────────
            print(f"{'Ep':>4}  {'TrainLoss':>9}  {'ValLoss':>9}  {'mIoU':>7}  {'PixAcc':>7}  {'Best':>7}  {'LR':>9}  {'Time':>6}")
            print('─' * 72)
            for r in data[-20:]:
                val_ran = r.get('val_ran', True)
                vl = fmt(r.get('val_loss'),   9) if val_ran else f'{"–skipped–":>9}'
                vm = fmt(r.get('val_miou'),   7) if val_ran else f'{"–":>7}'
                vp = fmt(r.get('pixel_acc'),  7) if val_ran else f'{"–":>7}'
                print(
                    f"{str(r.get('epoch','?')):>4}  "
                    f"{fmt(r.get('train_loss'), 9)}  "
                    f"{vl}  {vm}  {vp}  "
                    f"{fmt(r.get('best_miou'), 7)}  "
                    f"{fmt(r.get('lr'), 9, 6)}  "
                    f"{fmt(r.get('time_s'), 5, 0)}s"
                )

            valid_miou = [r['val_miou'] for r in data if r.get('val_miou') is not None]
            if valid_miou:
                best_ep = data[[r.get('val_miou') for r in data].index(max(valid_miou + [0]))]['epoch'] if valid_miou else '?'
                print(f'\n★ Best mIoU: {max(valid_miou):.4f}  |  Epochs done: {len(data)} / {EPOCHS}  |  Progress: {len(data)/EPOCHS*100:.0f}%')
            else:
                print(f'\nEpochs done: {len(data)} / {EPOCHS} — validation pending (runs every {VAL_EVERY} epochs)')

            # ── Live chart ────────────────────────────────────────
            epochs_x    = [r['epoch']      for r in data]
            train_loss  = [r.get('train_loss') for r in data]
            val_epochs  = [r['epoch']      for r in data if r.get('val_miou') is not None]
            val_miou_y  = [r['val_miou']   for r in data if r.get('val_miou') is not None]
            val_loss_y  = [r['val_loss']   for r in data if r.get('val_loss') is not None]

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.5))
            fig.suptitle(f'Training progress  — {len(data)}/{EPOCHS} epochs', fontsize=11)

            ax1.plot(epochs_x, train_loss, label='train loss', color='steelblue', linewidth=1.5)
            if val_loss_y:
                ax1.plot(val_epochs, val_loss_y, 'o-', label='val loss', color='tomato', linewidth=1.5, markersize=4)
            ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
            ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

            if val_miou_y:
                ax2.plot(val_epochs, val_miou_y, 'o-', color='mediumseagreen', linewidth=2, markersize=4)
                ax2.axhline(max(val_miou_y), color='mediumseagreen', linestyle='--', alpha=0.5,
                            label=f'best {max(val_miou_y):.4f}')
                ax2.legend()
            ax2.set_xlabel('Epoch'); ax2.set_ylabel('mIoU')
            ax2.set_title('Validation mIoU'); ax2.grid(alpha=0.3)

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f'Waiting for metrics... ({e})')
    else:
        print(f'Watching: {metrics_path}')
        print('No metrics.json yet — training is still in the first epoch.')

    ckpt_s = f'{ckpt_path.stat().st_size / 1e6:.1f} MB' if ckpt_path.exists() else 'not yet'
    best_s = f'{best_path.stat().st_size / 1e6:.1f} MB' if best_path.exists() else 'not yet'
    print(f'\nlast_checkpoint.pth : {ckpt_s}')
    print(f'best.pth            : {best_s}')
    print(f'\n(refreshing every 15 s — interrupt cell to stop)')

    time.sleep(15)


## Cell 10 — Verify saved files on Drive

In [ ]:
import pathlib, json

out = pathlib.Path(str(OUTPUT_DIR))
files = sorted(out.glob('*'))

print(f'Files in {out}:')
for f in files:
    size = f.stat().st_size / 1e6
    print(f'  {f.name:<40} {size:>8.2f} MB')

metrics_file = out / 'metrics.json'
if metrics_file.exists():
    data = json.loads(metrics_file.read_text())
    print(f'\nTotal epochs logged: {len(data)}')
    valid = [r for r in data if r.get('val_miou') is not None]
    if valid:
        best = max(valid, key=lambda r: r['val_miou'])
        print(f'Best epoch : {best["epoch"]}')
        print(f'  mIoU     : {best["val_miou"]:.4f}')
        print(f'  pixel_acc: {best.get("pixel_acc", 0):.4f}')
        print(f'  val_loss : {best.get("val_loss", 0):.4f}')


## Tips

| Situation | Fix |
|---|---|
| **OOM** | Lower `BATCH_SIZE` to 4, or `IMG_SIZE` to 384 |
| **Session reset** | Re-run cells 2 → 3 → 7 → 8 — auto-resumes from last checkpoint |
| **Live chart** | Open notebook in a second browser tab, connect to same runtime, run Cell 9 |
| **Faster training** | Switch `MODEL = 'resnet101'` for a stronger backbone (uses ~2× more memory) |
| **SegFormer** | Run Cell 5 first, then set `MODEL = 'segformer'` |
| **Export to iPhone** | Copy `best.pth` to your Mac, run: `python training/export_coreml.py --checkpoint training/output/best.pth` |
| **Existing checkpoint** | Make sure `OUTPUT_DIR` in Cell 2 points to the folder containing `last_checkpoint.pth` |
